In [20]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold

In [21]:
df_bt = pd.read_csv("files/high_cosine_similarity_records.csv",
                    usecols=["translated_welsh", "cefr_level"])\
         .rename(columns={"translated_welsh": "text"})

In [22]:
# Add missing columns 
df_bt["title"] = "Back-translated A1/A2 sample"
df_bt["lang"] = "cy"
df_bt["source_name"] = "back_translation_pipeline"
df_bt["format"] = "text"
df_bt["category"] = "general"
df_bt["license"] = "CC-BY-SA" 

In [23]:
df_bt

,text,cefr_level,title,lang,source_name,format,category,license
0,"Ac mae'r tywysog yn mynd i ffwrdd, yn ddryslyd.",A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1,"Dyma, i mi, y mae'r cariad yn drist a'r lleafa...",A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
2,Roedd y pumed yn rhyfedd iawn.,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
3,"Nid A oedd gofal llawer ar gyfer y nodau, roed...",A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
4,"I fod yn onest, erbyn hyn doeddwn i ddim wir w...",A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
...,...,...,...,...,...,...,...,...
1060,Gwnaeth farw ar 6 Hydref Hydref Hydref Hydref ...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1061,Y fersiwn cyntaf i' r fersiwn cyntaf.,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1062,Rhestr o wledydd eraill gan boblogaeth,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1063,"Roedd hi'n unig blentyn, geni yn Llundain.",A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA


In [24]:
CEFR_LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}
labels = np.array([label2id[l] for l in df_bt["cefr_level"]])

In [25]:
model_name = "./eurobert_cefr_welsh_b2/best_model"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

In [26]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [27]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [28]:
# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

In [29]:
hf_dataset=Dataset.from_pandas(df_bt.reset_index(drop=True))

In [30]:
hf_dataset

Dataset({
    features: ['text', 'cefr_level', 'title', 'lang', 'source_name', 'format', 'category', 'license'],
    num_rows: 1065
})

In [31]:
best_f1 = 0.0
best_trainer = None
best_tokenizer = None

for fold, (train_idx, val_idx) in enumerate(skf.split(hf_dataset, labels), start=1):
    print(f"\n Running Fold {fold}...")

    ds_train = hf_dataset.select(train_idx)
    ds_val = hf_dataset.select(val_idx)

    tok_train = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)
    tok_val = ds_val.map(preprocess, batched=True, remove_columns=ds_val.column_names)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(CEFR_LEVELS),trust_remote_code=True)

    args = TrainingArguments(
        output_dir=f"./eurobert_cefr_welsh_b2_only_DA/fold_{fold}",  
        num_train_epochs=3, 
        per_device_train_batch_size=2,              
        per_device_eval_batch_size=3,                
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_weighted_f1",
        greater_is_better=True,
        seed=42,
        learning_rate=3.6e-5,
        warmup_ratio=0.1,
        gradient_accumulation_steps=16,      
        optim="adamw_torch_fused",                   
        lr_scheduler_type="linear",                  
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    # Track best trainer
    if metrics["eval_weighted_f1"] > best_f1:
        best_f1 = metrics["eval_weighted_f1"]
        best_trainer = trainer
        best_tokenizer = tokenizer

    # Store fold metrics
    row = {
        "Fold": fold,
        "All CEFR Levels Precision": metrics.get("eval_weighted_precision", 0.0),
        "All CEFR Levels Recall": metrics.get("eval_weighted_recall", 0.0),
        "All CEFR Levels F1": metrics.get("eval_weighted_f1", 0.0),
    }
    for level in CEFR_LEVELS:
        row[f"{level} Precision"] = metrics.get(f"eval_{level}_precision", 0.0)
        row[f"{level} Recall"] = metrics.get(f"eval_{level}_recall", 0.0)
        row[f"{level} F1"] = metrics.get(f"eval_{level}_f1", 0.0)

    all_results.append(row)


 Running Fold 1...


Map: 100%|██████████| 213/213 [00:00<00:00, 16385.80 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_15260\3674199727.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.723500,0.431681,0.830986,0.826076,0.821877,0.830986,0.407407,0.354839,0.379310,0.892473,0.912088,0.902174,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.421100,0.394802,0.859155,0.837625,0.832826,0.859155,0.533333,0.258065,0.347826,0.883838,0.961538,0.921053,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.294400,0.440616,0.849765,0.830638,0.822681,0.849765,0.470588,0.258065,0.333333,0.882653,0.950549,0.915344,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



 Running Fold 2...


Map: 100%|██████████| 213/213 [00:00<00:00, 5133.34 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_15260\3674199727.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.754300,0.390427,0.859155,0.794067,0.738147,0.859155,0.000000,0.000000,0.000000,0.859155,1.000000,0.924242,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.445200,0.373552,0.868545,0.815704,0.885989,0.868545,1.000000,0.066667,0.125000,0.867299,1.000000,0.928934,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.283200,0.323852,0.877934,0.846060,0.864205,0.877934,0.750000,0.200000,0.315789,0.882927,0.989071,0.932990,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



 Running Fold 5...


Map: 100%|██████████| 213/213 [00:00<00:00, 17281.88 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_15260\3674199727.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.780600,0.553887,0.751174,0.772581,0.801954,0.751174,0.255319,0.400000,0.311688,0.891566,0.808743,0.848138,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.392700,0.363321,0.887324,0.857902,0.886001,0.887324,0.875000,0.233333,0.368421,0.887805,0.994536,0.938144,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.282600,0.355479,0.877934,0.861608,0.859959,0.877934,0.625000,0.333333,0.434783,0.898477,0.967213,0.931579,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [32]:
# Save best-performing model from all folds
final_path = "./eurobert_cefr_welsh_b2_only_DA/best_model"
best_trainer.save_model(final_path)
best_tokenizer.save_pretrained(final_path)
best_trainer.state.save_to_json(os.path.join(final_path, "trainer_state.json"))

In [33]:
# Convert to DataFrame
df = pd.DataFrame(all_results)

# Compute average row
average_row = df.drop(columns=["Fold"]).mean(numeric_only=True)
average_row["Fold"] = "Average"
df = pd.concat([df, pd.DataFrame([average_row])], ignore_index=True)

# Restructure columns
columns = [("Fold", "")] + [
    ("All CEFR Levels", "Precision"), ("All CEFR Levels", "Recall"), ("All CEFR Levels", "F1"),
    ("A1", "Precision"), ("A1", "Recall"), ("A1", "F1"),
    ("A2", "Precision"), ("A2", "Recall"), ("A2", "F1"),
    ("B1", "Precision"), ("B1", "Recall"), ("B1", "F1"),
    ("B2", "Precision"), ("B2", "Recall"), ("B2", "F1"),
    ("C1", "Precision"), ("C1", "Recall"), ("C1", "F1"),
    ("C2", "Precision"), ("C2", "Recall"), ("C2", "F1"),
]


df = df[[col[0] if col[1] == "" else f"{col[0]} {col[1]}" for col in columns]]
df.columns = pd.MultiIndex.from_tuples(columns)


In [34]:
df

Fold All CEFR Levels                            A1                      \
                 Precision    Recall        F1 Precision    Recall        F1   
0        1        0.832826  0.859155  0.837625  0.533333  0.258065  0.347826   
1        2        0.805932  0.845070  0.817883  0.384615  0.166667  0.232558   
2        3        0.846795  0.868545  0.850962  0.562500  0.300000  0.391304   
3        4        0.864205  0.877934  0.846060  0.750000  0.200000  0.315789   
4        5        0.859959  0.877934  0.861608  0.625000  0.333333  0.434783   
5  Average        0.841943  0.865728  0.842828  0.571090  0.251613  0.344452   

         A2                      ...   B1        B2                    C1  \
  Precision    Recall        F1  ...   F1 Precision Recall   F1 Precision   
0  0.883838  0.961538  0.921053  ...  0.0       0.0    0.0  0.0       0.0   
1  0.875000  0.956284  0.913838  ...  0.0       0.0    0.0  0.0       0.0   
2  0.893401  0.961749  0.926316  ...  0.0       0.0    0.0  0.0       0.0   
3  0.882927  0.989071  0.932990  ...  0.0       0.0    0.0  0.0       0.0   
4  0.898477  0.967213  0.931579  ...  0.0       0.0    0.0  0.0       0.0   
5  0.886729  0.967171  0.925155  ...  0.0       0.0    0.0  0.0       0.0   

                     C2              
  Recall   F1 Precision Recall   F1  
0    0.0  0.0       0.0    0.0  0.0  
1    0.0  0.0       0.0    0.0  0.0  
2    0.0  0.0       0.0    0.0  0.0  
3    0.0  0.0       0.0    0.0  0.0  
4    0.0  0.0       0.0    0.0  0.0  
5    0.0  0.0       0.0    0.0  0.0  

[6 rows x 22 columns]